## **RDD Transformations and Actions**

### **Spark 설정하기**

In [2]:
# [+] SparkConf, SparkContext 임포트
from pyspark import SparkConf, SparkContext

In [3]:
"""
    [+] SparkConf 객체 생성 및 설정
    - setMaster(), 마스터 노드 위치: "local"
    - 앱이름: "transformations_actions"
"""

conf = SparkConf()\
    .setMaster('local')\
    .setAppName('rdd-transformations-and-actions')

In [4]:
# [+] SparkContext 객체 생성
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 04:45:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/06 04:45:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
# Spark 전체 설정 값 확인
sc.getConf().getAll()

[('spark.master', 'local'),
 ('spark.driver.host', 'c582b8887d34'),
 ('spark.app.submitTime', '1791261944735'),
 ('spark.executor.id', 'driver'),
 ('spark.rdd.compress', 'True'),
 ('spark.executor.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jg

In [6]:
# SpackContext 객체 하나 더 생성하기(과연?)
sc = SparkContext(conf=conf)

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=rdd-transformations-and-actions, master=local) created by __init__ at /tmp/ipykernel_2628/824348036.py:2 

In [ ]:
# SparkContext는 프로그램 내에서 하나의 객체만 생성할 수 있다. (= Singleton 패턴)
# SparkContext 객체 해제하기
sc.stop()

In [ ]:
# 다시 생성하기
sc = SparkContext(conf=conf)

### **기본적인 액션 메서드들**
+ ```collect()```: 모든 파티션의 데이터들을 드라이버 노드로 수집 및 리스트로 반환
+ ```countByValue()```: 해당 값의 발생 빈도 계산
+ ```take(n)```: 처음 ```n```개의 값을 반환
+ ```first()```: 첫 번째 값을 반환
+ ```count()```: 값의 갯수를 계산

In [ ]:
"""
    [+] List로부터 RDD 객체 생성
    - SparkContext.parallelize()
    - 리스트 값: 
    "짜장면", "마라탕", "짬뽕", 
    "떡볶이", "쌀국수", "짬뽕", 
    "짜장면", "짜장면", "짜장면", 
    "라면", "우동", "라면"
"""

food_lst = [
    "짜장면", "마라탕", "짬뽕", 
    "떡볶이", "쌀국수", "짬뽕", 
    "짜장면", "짜장면", "짜장면", 
    "라면", "우동", "라면"]
       
# [+] lst로부터 RDD 객체 생성하기
foods = sc.parallelize(food_lst)

In [ ]:
# [+] foods RDD 내용 전체 출력
foods.collect()

In [ ]:
# [+] RDD 원소 값 별로 발생 빈도 계산
foods.countByValue()

In [ ]:
# [+] take(n): 처음 n개의 값을 출력
foods.take(5)

In [ ]:
# [+] first(): 첫 번째 값을 출력
foods.first()

In [7]:
# count(): 값 갯수 출력(중복된 원소 포함)
foods.count()

NameError: name 'foods' is not defined

In [8]:
# distinct(): unique한 원소로 구성된 RDD를 생성하는 Transformation

unique_foods = foods.distinct()
unique_foods.collect()

NameError: name 'foods' is not defined

### **기본적인 변환 메서드들**

In [9]:
# Transformations 은 지연 실행된다.
sc.parallelize([1, 2, 3]).map(lambda x: x + 2)

PythonRDD[1] at RDD at PythonRDD.scala:53

In [10]:
# Actions 를 만나면 즉시 실행된다.
sc.parallelize([1, 2, 3]).map(lambda x: x + 2).collect()

[3, 4, 5]

In [11]:
sc.parallelize([1, 2, 3]).map(lambda x: x * 2).collect()

[2, 4, 6]

In [12]:
movies = [
    "그린 북",
    "매트릭스",
    "토이 스토리",
    "캐스트 어웨이",
    "포드 V 페라리",
    "보헤미안 랩소디",
    "빽 투 더 퓨처",
    "반지의 제왕",
    "죽은 시인의 사회"
]

In [13]:
# [+] 리스트(movies) -> RDD 객체(moviesRDD) 생성
moviesRDD = sc.parallelize(movies)
moviesRDD.collect()

['그린 북',
 '매트릭스',
 '토이 스토리',
 '캐스트 어웨이',
 '포드 V 페라리',
 '보헤미안 랩소디',
 '빽 투 더 퓨처',
 '반지의 제왕',
 '죽은 시인의 사회']

In [14]:
# [+] flatMap을 이용한 Tokenizing
flatMovies = moviesRDD.flatMap(lambda x : x.split(" "))
flatMovies.collect()

['그린',
 '북',
 '매트릭스',
 '토이',
 '스토리',
 '캐스트',
 '어웨이',
 '포드',
 'V',
 '페라리',
 '보헤미안',
 '랩소디',
 '빽',
 '투',
 '더',
 '퓨처',
 '반지의',
 '제왕',
 '죽은',
 '시인의',
 '사회']

In [15]:
"""
    [+] flatMovies 에서 '매트릭스'를 제외
    - filter(lambda ...)
    - 저장할 객체: filteredMovies
"""

filteredMovies = flatMovies.filter(lambda x : x != '매트릭스')

In [16]:
filteredMovies.collect()

['그린',
 '북',
 '토이',
 '스토리',
 '캐스트',
 '어웨이',
 '포드',
 'V',
 '페라리',
 '보헤미안',
 '랩소디',
 '빽',
 '투',
 '더',
 '퓨처',
 '반지의',
 '제왕',
 '죽은',
 '시인의',
 '사회']

### **집합 연산 관련 변환 메서드들**
+ ```intersection(rdd)```: 교집합 계산
+ ```union(rdd)```: 합집합 계산
+ ```subtract(rdd)```: 차집합 계산

In [17]:
# [+] 두 개의 정수 리스트로부터 RDD 객체 생성(num1, num2)
num1 = sc.parallelize([1,2,3,4,5])
num2 = sc.parallelize([2,4,6,8])

In [18]:
# [+] 교집합 구하기
num1.intersection(num2).collect()

[2, 4]

In [19]:
# [+] 합집합 구하기
num1.union(num2).collect()

[1, 2, 3, 4, 5, 2, 4, 6, 8]

In [20]:
# [+] 차집합 구하기
num1.subtract(num2).collect()

[1, 3, 5]

### **랜덤 샘플링 함수**
+ ```sample(withReplacement, fraction)```: 원소를 랜덤 샘플링하여 새로운 RDD를 생성하는 narrow transformation
    + ```withReplacement```, bool[optional]: 중복 샘플링 허용 여부
    + ```fraction```, float[optional]: 샘플링 확률[0~1], 확률이므로 정확한 사이즈가 보장되지 않음
    + ```seed```: 결과 재현(reproducibility)을 위한 설정 값

In [21]:
# [+] 합집합 생성(num1 + num2 -> numUnion)
numUnion = num1.union(num2)
numUnion.collect()

[1, 2, 3, 4, 5, 2, 4, 6, 8]

In [22]:
# [+] 샘플링: 중복 허용, 50%
numUnion.sample(True, .5).collect()

[5, 5, 2]

In [23]:
# [+] seed 파라미터를 설정하면 동일한 결과를 반복적으로 얻을 수 있다.(시 고정)
numUnion.sample(True, .5, seed=5).collect()

[1, 4, 6]

### **그룹핑 연산**
+ ```groupBy()```: 특정 기준(함수로 정의)으로 그룹핑을 수행하는 wide transformation

In [24]:
students = sc.parallelize([
    "이수경","이학선","이준솔",
    "이채원","윤지원","김태근",
    "서지유","이다영","정승윤",
    "박민규","이예소","이영림",
    "정승준","이용희","유형원",
    "박병규","방유리","박진영",
    "장민혁","김도훈"
])

In [25]:
# [+] 원소의 첫 번째 값을 기준으로 그룹핑
studentGroup = students.groupBy(lambda x : x[0])
studentGroup.collect()

[('이', <pyspark.resultiterable.ResultIterable at 0x77df59211b50>),
 ('윤', <pyspark.resultiterable.ResultIterable at 0x77df59207950>),
 ('김', <pyspark.resultiterable.ResultIterable at 0x77df59222510>),
 ('서', <pyspark.resultiterable.ResultIterable at 0x77df59212e10>),
 ('정', <pyspark.resultiterable.ResultIterable at 0x77df592224e0>),
 ('박', <pyspark.resultiterable.ResultIterable at 0x77df59221fd0>),
 ('유', <pyspark.resultiterable.ResultIterable at 0x77df59222060>),
 ('방', <pyspark.resultiterable.ResultIterable at 0x77df59222630>),
 ('장', <pyspark.resultiterable.ResultIterable at 0x77df592222a0>)]

In [26]:
# collect() 결과를 리스트 객체(res)에 저장
res = studentGroup.collect()
res

[('이', <pyspark.resultiterable.ResultIterable at 0x77df380fc770>),
 ('윤', <pyspark.resultiterable.ResultIterable at 0x77df592124e0>),
 ('김', <pyspark.resultiterable.ResultIterable at 0x77df59212960>),
 ('서', <pyspark.resultiterable.ResultIterable at 0x77df59212a50>),
 ('정', <pyspark.resultiterable.ResultIterable at 0x77df59211c10>),
 ('박', <pyspark.resultiterable.ResultIterable at 0x77df59213230>),
 ('유', <pyspark.resultiterable.ResultIterable at 0x77df592138f0>),
 ('방', <pyspark.resultiterable.ResultIterable at 0x77df59210bf0>),
 ('장', <pyspark.resultiterable.ResultIterable at 0x77df592211f0>)]

In [27]:
res[0][1]

In [28]:
# ResultIterable 객체는 list로 쉽게 변환 가능하다.
list(res[0][1])

['이수경', '이학선', '이준솔', '이채원', '이다영', '이예소', '이영림', '이용희']

In [29]:
# [+] for 문을 이용하여 구조적으로 결과를 출력
for (k, v) in res:
    print(k, list(v))

이 ['이수경', '이학선', '이준솔', '이채원', '이다영', '이예소', '이영림', '이용희']
윤 ['윤지원']
김 ['김태근', '김도훈']
서 ['서지유']
정 ['정승윤', '정승준']
박 ['박민규', '박병규', '박진영']
유 ['유형원']
방 ['방유리']
장 ['장민혁']


In [30]:
# groupBy()는 모드 연산과 같이 수학 연산에도 사용될 수 있다.
nums = sc.parallelize([1,2,3,4,5,6,7,8,9,10])

In [31]:
# 홀수/짝수로 그룹핑
nums.groupBy(lambda x: x % 2).collect()

[(1, <pyspark.resultiterable.ResultIterable at 0x77df592133b0>),
 (0, <pyspark.resultiterable.ResultIterable at 0x77df380ff7d0>)]

In [32]:
res = nums.groupBy(lambda x: x % 2).collect()

In [33]:
# [+] '1' 그룹에 해당되는 원소 리스트 출력하기
list(res[0][1])

[1, 3, 5, 7, 9]

In [34]:
# [+] '0' 그룹에 해당되는 원소 리스트 출력하기
list(res[1][1])

[2, 4, 6, 8, 10]